In [1]:
print("hello")

hello


In [2]:
print("Python is working!")

Python is working!


In [3]:
import pandas as pd

df = pd.read_csv(
    "../data/raw/twcs.csv",
    nrows=10000
)

print("Loaded!")

Loaded!


In [4]:
df.shape

(10000, 7)

In [5]:
df.columns.tolist()

['tweet_id',
 'author_id',
 'inbound',
 'created_at',
 'text',
 'response_tweet_id',
 'in_response_to_tweet_id']

In [6]:
df.head(10)

,tweet_id,author_id,inbound,created_at,text,response_tweet_id,in_response_to_tweet_id
0,1,sprintcare,False,Tue Oct 31 22:10:47 +0000 2017,@115712 I understand. I would like to assist y...,2,3.0
1,2,115712,True,Tue Oct 31 22:11:45 +0000 2017,@sprintcare and how do you propose we do that,NaN,1.0
2,3,115712,True,Tue Oct 31 22:08:27 +0000 2017,@sprintcare I have sent several private messag...,1,4.0
3,4,sprintcare,False,Tue Oct 31 21:54:49 +0000 2017,@115712 Please send us a Private Message so th...,3,5.0
4,5,115712,True,Tue Oct 31 21:49:35 +0000 2017,@sprintcare I did.,4,6.0
5,6,sprintcare,False,Tue Oct 31 21:46:24 +0000 2017,@115712 Can you please send us a private messa...,"5,7",8.0
6,8,115712,True,Tue Oct 31 21:45:10 +0000 2017,@sprintcare is the worst customer service,"9,6,10",NaN
7,11,sprintcare,False,Tue Oct 31 22:10:35 +0000 2017,@115713 This is saddening to hear. Please shoo...,NaN,12.0
8,12,115713,True,Tue Oct 31 22:04:47 +0000 2017,@sprintcare You gonna magically change your co...,"11,13,14",15.0
9,15,sprintcare,False,Tue Oct 31 20:03:31 +0000 2017,@115713 We understand your concerns and we'd l...,12,16.0


In [7]:
df["inbound"].value_counts()

inbound
True     5503
False    4497
Name: count, dtype: int64

In [ ]:
df["author_id"].nunique()
#How many different people/accounts sent these tweets(includes companies and custoemrs)

3039

In [ ]:
df[df["inbound"] == False]["author_id"].value_counts().head(20)
#Take only company replies, count how many tweets each account sent, and show the top 20.

author_id
ChipotleTweets     433
AmazonHelp         410
AppleSupport       318
Uber_Support       189
comcastcares       163
British_Airways    158
VerizonSupport     154
Delta              154
AskPlayStation     153
TMobileHelp        142
SpotifyCares       136
SouthwestAir       127
hulu_support       119
AmericanAir        112
AdobeCare          107
sprintcare          97
Tesco               97
XboxSupport         95
Ask_Spectrum        90
TacoBellTeam        90
Name: count, dtype: int64

In [ ]:
#how many tweets belong to each brand in the full dataset.
brand_counts = {}

for chunk in pd.read_csv("../data/raw/twcs.csv", chunksize=100000):
    company_tweets = chunk[chunk["inbound"] == False]
    
    counts = company_tweets["author_id"].value_counts()
    
    for brand, count in counts.items():
        brand_counts[brand] = brand_counts.get(brand, 0) + count

brand_counts = pd.Series(brand_counts).sort_values(ascending=False)

brand_counts.head(30)

AmazonHelp         169840
AppleSupport       106860
Uber_Support        56270
SpotifyCares        43265
Delta               42253
Tesco               38573
AmericanAir         36764
TMobileHelp         34317
comcastcares        33031
British_Airways     29361
SouthwestAir        28977
VirginTrains        27817
Ask_Spectrum        25860
XboxSupport         24557
sprintcare          22381
hulu_support        21872
sainsburys          19466
GWRHelp             19364
AskPlayStation      19098
ChipotleTweets      18749
VerizonSupport      17966
UPSHelp             17817
ATVIAssist          17650
O2                  16212
Safaricom_Care      16077
idea_cares          15724
AskTarget           13218
AirAsiaSupport      12829
BofA_Help           12683
SW_Help             12231
dtype: int64

In [ ]:
#For each brand, how many company tweets are actually replies to a previous tweet?.. How many are real company
# replies to the prev any tweets(its not only customers)

#Because 
#inbound = False
#only tells us that the current tweet is from the company.

#And in_response_to_tweet_id != NaN
#tells us that it is replying to something.
#it does not yet prove that the previous tweet was from a customer.

top_brands = [
    "AmazonHelp",
    "AppleSupport",
    "Uber_Support",
    "SpotifyCares",
    "Delta"
]

resolution_counts = {}

for chunk in pd.read_csv("../data/raw/twcs.csv", chunksize=100000):
    for brand in top_brands:
        brand_tweets = chunk[
            (chunk["author_id"] == brand) &
            (chunk["inbound"] == False)
        ]

        # Company tweets that are replies to another tweet
        replied = brand_tweets["in_response_to_tweet_id"].notna()

        resolution_counts[brand] = (
            resolution_counts.get(brand, 0) + replied.sum()
        )

pd.Series(resolution_counts).sort_values(ascending=False)

AmazonHelp      169287
AppleSupport    106719
Uber_Support     56261
SpotifyCares     43243
Delta            42197
dtype: int64

Earlier we found:

AmazonHelp = 169,840 company tweets

Now:

169,287 of them are replies

So almost all AmazonHelp tweets in the dataset are actually part of a reply/conversation.

In [13]:
#How many company replies are directly replying to customer tweets?
top_brands = [
    "AmazonHelp",
    "AppleSupport",
    "Uber_Support",
    "SpotifyCares",
    "Delta"
]

customer_tweet_ids = set()
company_replies_to_customer = {brand: 0 for brand in top_brands}

# First: collect IDs of customer tweets
for chunk in pd.read_csv("../data/raw/twcs.csv", chunksize=100000):
    customer_tweets = chunk[chunk["inbound"] == True]
    customer_tweet_ids.update(customer_tweets["tweet_id"].astype(str))

# Second: check which company tweets reply directly to customers
for chunk in pd.read_csv("../data/raw/twcs.csv", chunksize=100000):
    for brand in top_brands:
        brand_tweets = chunk[
            (chunk["author_id"] == brand) &
            (chunk["inbound"] == False)
        ]

        for parent_id in brand_tweets["in_response_to_tweet_id"].dropna():
            if str(int(parent_id)) in customer_tweet_ids:
                company_replies_to_customer[brand] += 1

pd.Series(company_replies_to_customer).sort_values(ascending=False)

AmazonHelp      168814
AppleSupport    106646
Uber_Support     56160
SpotifyCares     43092
Delta            42114
dtype: int64

In [14]:
amazon = df[df["author_id"] == "AmazonHelp"]

amazon[["tweet_id", "inbound", "text", 
        "response_tweet_id", "in_response_to_tweet_id"]].head(20)

,tweet_id,inbound,text,response_tweet_id,in_response_to_tweet_id
181,269,False,@115770 こんにちは、アマゾン公式です。Fire TV Stickが見れないというのは...,"270,271",272.0
184,273,False,@115770 カスタマーサービスにてお問い合わせ済みとのことで、お手数をおかけいたしました...,274,271.0
186,275,False,@115770 恐れ入ります。至らない点も多々あるかとは存じますが、今後ともどうぞよろしくお...,NaN,274.0
234,324,False,@115792 ご不便をおかけしております。アプリをご利用でしょうか。強制停止&gt;端末の...,NaN,325.0
321,615,False,@115820 I'm sorry we've let you down! Without ...,616,617.0
323,618,False,@115820 We'd like to take a further look into ...,619,616.0
326,620,False,@115822 I am unable to affect your account via...,NaN,621.0
328,622,False,"@115824 Hi, wir erhalten die Filme/Serien so v...",623,624.0
330,625,False,@115824 Wir haben zu danken. Schönen Abend noc...,NaN,623.0
332,626,False,@115826 I'm sorry for the wait. You'll receive...,627,628.0


In [ ]:
tweet_lookup = df.set_index("tweet_id").to_dict("index")
#tweet ID → all information about that tweet
# Create a dictionary where each tweet_id maps to all information about that tweet.
# This lets us quickly find a tweet using its tweet_id when reconstructing conversations.

In [18]:
tweet_lookup[2]["text"]

'@sprintcare and how do you propose we do that'

In [19]:
parent_id = df.loc[df["tweet_id"] == 2, "in_response_to_tweet_id"].iloc[0]

print("Tweet 2 replied to:", parent_id)
print("Parent message:", tweet_lookup[int(parent_id)]["text"])

Tweet 2 replied to: 1.0
Parent message: @115712 I understand. I would like to assist you. We would need to get you into a private secured link to further assist.


In [20]:
tweet_id = 2

while tweet_id in tweet_lookup:
    tweet = tweet_lookup[tweet_id]

    print("Tweet ID:", tweet_id)
    print("Author:", tweet["author_id"])
    print("Customer?" , tweet["inbound"])
    print("Text:", tweet["text"])
    print("-" * 80)

    parent_id = tweet["in_response_to_tweet_id"]

    if pd.isna(parent_id):
        break

    tweet_id = int(parent_id)

Tweet ID: 2
Author: 115712
Customer? True
Text: @sprintcare and how do you propose we do that
--------------------------------------------------------------------------------
Tweet ID: 1
Author: sprintcare
Customer? False
Text: @115712 I understand. I would like to assist you. We would need to get you into a private secured link to further assist.
--------------------------------------------------------------------------------
Tweet ID: 3
Author: 115712
Customer? True
Text: @sprintcare I have sent several private messages and no one is responding as usual
--------------------------------------------------------------------------------
Tweet ID: 4
Author: sprintcare
Customer? False
Text: @115712 Please send us a Private Message so that we can further assist you. Just click ‘Message’ at the top of your profile.
--------------------------------------------------------------------------------
Tweet ID: 5
Author: 115712
Customer? True
Text: @sprintcare I did.
-------------------------------

In [ ]:
# Extract tweets from our top candidate support brands.
# We read the large dataset in chunks to avoid loading the entire file into memory.
# Only tweets belonging to the selected brands are kept for further analysis.
top_brands = [
    "AmazonHelp",
    "AppleSupport",
    "Uber_Support",
    "SpotifyCares",
    "Delta"
]

brand_data = []

for chunk in pd.read_csv("../data/raw/twcs.csv", chunksize=100000):
    matching = chunk[
        chunk["author_id"].isin(top_brands)
    ]
    
    brand_data.append(matching)

brand_df = pd.concat(brand_data, ignore_index=True)
#brand_df is simply the new DataFrame we created to store tweets from our 5 candidate brands.

print("Rows:", len(brand_df))
print("Columns:", brand_df.columns.tolist())

Rows: 418488
Columns: ['tweet_id', 'author_id', 'inbound', 'created_at', 'text', 'response_tweet_id', 'in_response_to_tweet_id']


In [22]:
brand_df["author_id"].value_counts()

author_id
AmazonHelp      169840
AppleSupport    106860
Uber_Support     56270
SpotifyCares     43265
Delta            42253
Name: count, dtype: int64

In [23]:
brand_df.groupby("author_id")["inbound"].value_counts()

author_id     inbound
AmazonHelp    False      169840
AppleSupport  False      106860
Delta         False       42253
SpotifyCares  False       43265
Uber_Support  False       56270
Name: count, dtype: int64

In [26]:
# Check whether each brand reply is linked to the customer tweet it is responding to.
brand_df["in_response_to_tweet_id"].notna().value_counts()

in_response_to_tweet_id
True     417707
False       781
Name: count, dtype: int64

417,707 brand tweets have a in_response_to_tweet_id → they are replies to another tweet.
Only 781 don't have a parent tweet.

In [27]:
# Get the tweet IDs that each brand reply is responding to.
parent_ids = brand_df["in_response_to_tweet_id"].dropna().astype(int)

print("Number of parent tweet IDs:", len(parent_ids))
print("Unique parent tweet IDs:", parent_ids.nunique())

Number of parent tweet IDs: 417707
Unique parent tweet IDs: 395356


In [28]:
# Retrieve the tweets that our brand replies were responding to.
# These parent tweets will let us connect each company response to the
# original customer message.

parent_ids_set = set(parent_ids)

parent_data = []

for chunk in pd.read_csv("../data/raw/twcs.csv", chunksize=100000):
    matching = chunk[
        chunk["tweet_id"].isin(parent_ids_set)
    ]
    
    parent_data.append(matching)

parent_df = pd.concat(parent_data, ignore_index=True)

print("Parent tweets found:", len(parent_df))
print("Unique parent tweet IDs:", parent_df["tweet_id"].nunique())

Parent tweets found: 394673
Unique parent tweet IDs: 394673


brand_df
418,488 brand tweets

to:

parent_df
customer tweets that those brand tweets responded to

In [30]:
# Check whether the parent tweets are customer messages or company replies.
parent_df["inbound"].value_counts()

inbound
True     394483
False       190
Name: count, dtype: int64

394,483 parent tweets are customer messages.
Only 190 are company tweets.
That means almost all of our brand responses point directly to customer messages.

In [31]:
# Connect each brand response to the customer tweet it is responding to.
# This gives us the historical customer → brand interactions
# that will be used for intent discovery and retrieval.

customer_df = parent_df[parent_df["inbound"] == True].copy()

pairs_df = brand_df.merge(
    customer_df[["tweet_id", "author_id", "text", "created_at"]],
    left_on="in_response_to_tweet_id",
    right_on="tweet_id",
    suffixes=("_brand", "_customer")
)

print("Customer-brand pairs:", len(pairs_df))
print("Unique customer tweets:", pairs_df["tweet_id_customer"].nunique())

Customer-brand pairs: 416826
Unique customer tweets: 394483


416,826 rows = individual customer → brand response relationships.
394,483 unique customers' tweets = distinct customer messages that received at least one brand response.
The difference (~22k) means some customer tweets received multiple brand responses.

In [32]:
# Compare the number of customer-support interactions available for each brand.
pairs_df["author_id_brand"].value_counts()

author_id_brand
AmazonHelp      168814
AppleSupport    106646
Uber_Support     56160
SpotifyCares     43092
Delta            42114
Name: count, dtype: int64

In [33]:
# Count the number of unique customer conversations for each brand.
# A conversation is identified by the customer tweet that the brand replied to.

pairs_df.groupby("author_id_brand")["tweet_id_customer"].nunique()

author_id_brand
AmazonHelp      154976
AppleSupport    106623
Delta            36134
SpotifyCares     41585
Uber_Support     55182
Name: tweet_id_customer, dtype: int64

In [34]:
# Inspect a few real Amazon customer → brand interactions.
amazon_pairs = pairs_df[
    pairs_df["author_id_brand"] == "AmazonHelp"
]

amazon_pairs[
    ["text_customer", "text_brand"]
].head(20)

,text_customer,text_brand
0,amazonのfireTVstickが見れない😢,@115770 こんにちは、アマゾン公式です。Fire TV Stickが見れないというのは...
1,@AmazonHelp 電話で対応してもらいましたが改良されませんでした。\n保証期間も過ぎ...,@115770 カスタマーサービスにてお問い合わせ済みとのことで、お手数をおかけいたしました...
2,@AmazonHelp こちらこそありがとうございました。,@115770 恐れ入ります。至らない点も多々あるかとは存じますが、今後ともどうぞよろしくお...
3,amazonプライムビデオ、再生エラーが多いです,@115792 ご不便をおかけしております。アプリをご利用でしょうか。強制停止&gt;端末の...
5,Way to drop the ball on customer service @1158...,@115820 I'm sorry we've let you down! Without ...
6,@AmazonHelp 3 different people have given 3 di...,@115820 We'd like to take a further look into ...
7,@115823 I want my amazon payments account CLOS...,@115822 I am unable to affect your account via...
8,"@115825 also, beim Addams Family-Film in Prime...","@115824 Hi, wir erhalten die Filme/Serien so v..."
9,"@AmazonHelp Okay, danke für die Info",@115824 Wir haben zu danken. Schönen Abend noc...
10,@115828 How about you guys figure out my Xbox ...,@115826 I'm sorry for the wait. You'll receive...


In [35]:
# Keep only AmazonHelp interactions for the support agent.
# We will use these historical interactions for conversation analysis,
# intent discovery, retrieval, and evaluation.

amazon_pairs = pairs_df[
    pairs_df["author_id_brand"] == "AmazonHelp"
].copy()

print("Amazon interactions:", len(amazon_pairs))
print("Unique customer tweets:", amazon_pairs["tweet_id_customer"].nunique())

Amazon interactions: 168814
Unique customer tweets: 154976


In [36]:
# Keep only AmazonHelp interactions for the support agent.
# We will use these historical interactions for conversation analysis,
# intent discovery, retrieval, and evaluation.

amazon_pairs = pairs_df[
    pairs_df["author_id_brand"] == "AmazonHelp"
].copy()

print("Amazon interactions:", len(amazon_pairs))
print("Unique customer tweets:", amazon_pairs["tweet_id_customer"].nunique())

Amazon interactions: 168814
Unique customer tweets: 154976


In [37]:
# Count how many AmazonHelp responses are associated with each customer tweet.
# This helps us understand how often a customer issue receives multiple responses.

conversation_depth = (
    amazon_pairs
    .groupby("tweet_id_customer")
    .size()
    .value_counts()
    .sort_index()
)

print(conversation_depth)

1    142700
2     10785
3      1429
4        55
5         5
6         2
Name: count, dtype: int64


Out of 154,976 unique customer tweets:

142,700 received 1 Amazon response
10,785 received 2 responses
1,429 received 3 responses
62 had 4+ responses

In [38]:
# Select one Amazon customer interaction to inspect.
sample_customer_id = amazon_pairs["tweet_id_customer"].iloc[0]

print("Customer tweet ID:", sample_customer_id)

Customer tweet ID: 272


In [39]:
# Follow the conversation backward from the selected customer tweet.
# Each in_response_to_tweet_id points to the message this tweet replied to.

tweet_id = sample_customer_id

while tweet_id in tweet_lookup:
    tweet = tweet_lookup[tweet_id]

    print("Tweet ID:", tweet_id)
    print("Author:", tweet["author_id"])
    print("Customer?", tweet["inbound"])
    print("Text:", tweet["text"])
    print("-" * 80)

    parent_id = tweet["in_response_to_tweet_id"]

    if pd.isna(parent_id):
        break

    tweet_id = int(parent_id)

Tweet ID: 272
Author: 115770
Customer? True
Text: amazonのfireTVstickが見れない😢
--------------------------------------------------------------------------------


In [40]:
# Check which tweet the selected customer message replied to.
tweet_272 = tweet_lookup[sample_customer_id]

print("Parent tweet ID:", tweet_272["in_response_to_tweet_id"])

Parent tweet ID: nan


In [41]:
# Check which tweet(s) AmazonHelp sent in response to the customer.
print("Response tweet ID(s):", tweet_272["response_tweet_id"])

Response tweet ID(s): 269


In [42]:
# Display the customer message and the AmazonHelp response side by side.

customer = tweet_lookup[272]
response = tweet_lookup[269]

print("CUSTOMER:")
print(customer["text"])

print("\nAMAZONHELP:")
print(response["text"])

CUSTOMER:
amazonのfireTVstickが見れない😢

AMAZONHELP:
@115770 こんにちは、アマゾン公式です。Fire TV Stickが見れないというのは、どのような状況でしょうか。一般的なトラブルシューティングを記載したヘルプがございますので、ご参照ください。https://t.co/2pbG55qJ7h ET


brand_df → AmazonHelp tweets
parent_df → customer tweets that Amazon replied to
amazon_pairs → customer → Amazon response pairs

In [43]:
# Combine the customer message and AmazonHelp response into one clean dataset.
# Each row represents one historical customer → AmazonHelp interaction.

amazon_history = amazon_pairs[
    [
        "tweet_id_customer",
        "author_id_customer",
        "text_customer",
        "created_at_customer",
        "tweet_id_brand",
        "text_brand",
        "created_at_brand"
    ]
].copy()

# Rename the columns to make them easier to understand and use later.
amazon_history = amazon_history.rename(columns={
    "tweet_id_customer": "customer_tweet_id",
    "author_id_customer": "customer_id",
    "text_customer": "customer_message",
    "created_at_customer": "customer_time",
    "tweet_id_brand": "brand_tweet_id",
    "text_brand": "brand_response",
    "created_at_brand": "brand_time"
})

print("Rows:", len(amazon_history))
print("Columns:", amazon_history.columns.tolist())

Rows: 168814
Columns: ['customer_tweet_id', 'customer_id', 'customer_message', 'customer_time', 'brand_tweet_id', 'brand_response', 'brand_time']


In [44]:
# Check whether any customer messages or AmazonHelp responses are missing.
# We need both sides of the interaction for training and retrieval.

print("Missing customer messages:")
print(amazon_history["customer_message"].isna().sum())

print("\nMissing Amazon responses:")
print(amazon_history["brand_response"].isna().sum())

Missing customer messages:
0

Missing Amazon responses:
0


In [45]:
# Check for completely duplicated customer → AmazonHelp interactions.
# Duplicate rows could distort our intent counts and retrieval results.

print("Duplicate rows:", amazon_history.duplicated().sum())

Duplicate rows: 0


In [46]:
# Display a sample of historical Amazon customer messages and responses.
# This helps us understand the types of support issues in the dataset
# before defining our intent categories.

sample = amazon_history[
    ["customer_message", "brand_response"]
].sample(20, random_state=42)

pd.set_option("display.max_colwidth", 300)

sample

,customer_message,brand_response
242736,@119959 @115828 I preordered this game 14 months ago just to be sent it for the wrong console. FML... https://t.co/EMBHygMbOL,@520984 I'm sorry for the mix-up! Please contact us so we can look into this and review options: https://t.co/hApLpMlfHN ^WT
95602,@AmazonHelp 2ème colis qui devait arriver aujourd'hui et ne le sera pas...C'est quoi votre engagement Prime déjà?,@242837 Que dit le suivi de votre colis s'il vous plaît ? ^MA
116485,@115821 are y’all ever gonna transfer my $50 gc on an account you closed without telling me...? jw,"@315479 To confirm, has your account been closed or has it been locked pending account information verification? ^EZ"
293784,@115821 dudes. You refunded me for the wrong item. I got my wine. Just not my #reesespeanutbuttercups the wine cost more! I only needed $4!,"@622478 We greatly appreciate your honesty! Please provide your info here: https://t.co/e1bGy9Xhvk, so we look into this for you. ^JN"
57332,@AmazonHelp Fuck u then why the hell u r having @AmazonHelp Twitter handle,@210361 Usually the products are delivered by the estimated delivery date. Could you let me know if we've missed it? ^AU
53547,@AmazonHelp I had a scheduled furniture assembly yesterday which never happened. Upon calling the customer care they could not help much apart.,"@206017 Apologies for the delay, I'd like to look into this, kindly drop in your details here:https://t.co/beaaDm0muc and 1/2 ^EM"
312372,@115821 don’t tell me I got guaranteed delivery and then change the date the day I’m supposed to get everything 😤,@229000 I'm sorry to hear about this! Have you received an e-mail regarding any type of delay? ^VB
66746,@AmazonHelp Amazon shipping.,@119036 Thanks for confirming with us! We'd like to look into this with you in more detail. Please reach out to us here: https://t.co/JzP7hlA23B so we can investigate! ^VB
261716,@AmazonHelp just had chat with a representative and got assured delivery by 5 pm today. Lets see what happens....,@555171 Thanks. Do keep us posted. ^HK
181385,@AmazonHelp My lights all appear here but my Dots do not. Should this be where I select them to add? https://t.co/L7OX9LKLlD,"@421770 Hello, Alanna! We want to help! Contact us here so we can get your Smart Home set up: https://t.co/hApLpMlfHN ^EA"


In [47]:
# Combine AmazonHelp tweets and the customer tweets they responded to.
# This gives us the relevant tweets needed to reconstruct conversations
# without loading the entire 3M+ tweet dataset into memory.

amazon_tweets = brand_df[
    brand_df["author_id"] == "AmazonHelp"
].copy()

customer_tweets = parent_df[
    parent_df["inbound"] == True
].copy()

conversation_tweets = pd.concat(
    [amazon_tweets, customer_tweets],
    ignore_index=True
)

# Remove any duplicate tweet IDs.
# A tweet may appear more than once because of the way we extracted the data.
conversation_tweets = conversation_tweets.drop_duplicates(
    subset="tweet_id"
)

print("Tweets available for reconstruction:", len(conversation_tweets))
print("Unique tweet IDs:", conversation_tweets["tweet_id"].nunique())

Tweets available for reconstruction: 564323
Unique tweet IDs: 564323


In [48]:
# Create a lookup from tweet ID to the complete tweet information.
# This lets us quickly find the parent tweet when reconstructing a conversation.

conversation_lookup = (
    conversation_tweets
    .set_index("tweet_id")
    .to_dict("index")
)

print("Tweets in lookup:", len(conversation_lookup))

Tweets in lookup: 564323


In [50]:
# Start from customer tweet 272 and follow the conversation backward.
# Each tweet points to the tweet it replied to using in_response_to_tweet_id.

tweet_id = 272
conversation = []

while tweet_id in conversation_lookup:
    tweet = conversation_lookup[tweet_id]

    conversation.append((tweet_id, tweet))

    parent_id = tweet["in_response_to_tweet_id"]

    # Stop when this tweet is the beginning of the conversation.
    if pd.isna(parent_id):
        break

    tweet_id = int(parent_id)

print("Number of tweets in conversation:", len(conversation))

# Print the conversation from oldest to newest.
for tweet_id, tweet in reversed(conversation):
    print("Tweet ID:", tweet_id)
    print("Author:", tweet["author_id"])
    print("Customer?", tweet["inbound"])
    print("Text:", tweet["text"])
    print("-" * 80)

Number of tweets in conversation: 1
Tweet ID: 272
Author: 115770
Customer? True
Text: amazonのfireTVstickが見れない😢
--------------------------------------------------------------------------------


In [51]:
# Follow the response link from the customer tweet to AmazonHelp.
# This checks that we can move forward through the conversation.

tweet_id = 272
tweet = conversation_lookup[tweet_id]

response_ids = tweet["response_tweet_id"]

print("Customer message:")
print(tweet["text"])

print("\nAmazon response tweet ID(s):")
print(response_ids)

Customer message:
amazonのfireTVstickが見れない😢

Amazon response tweet ID(s):
269


In [52]:
# Retrieve the AmazonHelp response using the response tweet ID.
# This lets us see the customer message and its direct Amazon response together.

response_id = int(response_ids)

amazon_response = conversation_lookup[response_id]

print("Customer:")
print(tweet["text"])

print("\nAmazonHelp:")
print(amazon_response["text"])

Customer:
amazonのfireTVstickが見れない😢

AmazonHelp:
@115770 こんにちは、アマゾン公式です。Fire TV Stickが見れないというのは、どのような状況でしょうか。一般的なトラブルシューティングを記載したヘルプがございますので、ご参照ください。https://t.co/2pbG55qJ7h ET


In [53]:
# Find a customer tweet that received multiple AmazonHelp responses.
# This gives us a better example of a multi-turn support interaction.

multi_reply_customer_ids = (
    amazon_history
    .groupby("customer_tweet_id")
    .size()
)

multi_reply_customer_ids = multi_reply_customer_ids[
    multi_reply_customer_ids >= 2
]

print("Customer tweets with 2+ Amazon responses:",
      len(multi_reply_customer_ids))

# Select one example.
sample_multi_id = multi_reply_customer_ids.index[0]

print("Example customer tweet ID:", sample_multi_id)

Customer tweets with 2+ Amazon responses: 12276
Example customer tweet ID: 686


In [54]:
# Show all AmazonHelp responses linked to customer tweet 686.
# This helps us see how the same customer issue can receive multiple replies.

example = amazon_history[
    amazon_history["customer_tweet_id"] == sample_multi_id
]

example[
    ["customer_message", "brand_response"]
]

,customer_message,brand_response
32,@AmazonHelp Already contacted 3-4 times in the last month. But all I got was assurance that matter is escalated &amp; new status will be provided in 1 day!,@115849 Please share your details here:https://t.co/GIJyeYqKE0 and I'll get back to you. 2/2 ^HD
141311,@AmazonHelp Already contacted 3-4 times in the last month. But all I got was assurance that matter is escalated &amp; new status will be provided in 1 day!,"@115849 I understand your concern, James. Allow me to take a closer look. 1/2 ^HD"


In [55]:
# Inspect the original customer tweet and the Amazon tweets connected to it.
# This helps us understand how Twitter splits multi-part replies.

customer_686 = conversation_lookup[686]

print("CUSTOMER:")
print(customer_686["text"])

print("\nAMAZON RESPONSE ID(S):")
print(customer_686["response_tweet_id"])

CUSTOMER:
@AmazonHelp Already contacted 3-4 times in the last month. But all I got was assurance that matter is escalated &amp; new status will be provided in 1 day!

AMAZON RESPONSE ID(S):
684,688


In [56]:
# Retrieve the Amazon tweets connected to customer tweet 686.
# We inspect their IDs, timestamps, and text to understand their order.

for response_id in [684, 688]:
    response = conversation_lookup[response_id]

    print("Tweet ID:", response_id)
    print("Time:", response["created_at"])
    print("Text:", response["text"])
    print("-" * 80)

Tweet ID: 684
Time: Tue Oct 31 22:17:00 +0000 2017
Text: @115849 Please share your details here:https://t.co/GIJyeYqKE0 and I'll get back to you. 2/2 ^HD
--------------------------------------------------------------------------------
Tweet ID: 688
Time: Tue Oct 31 22:13:26 +0000 2017
Text: @115849 I understand your concern, James. Allow me to take a closer look. 1/2 ^HD
--------------------------------------------------------------------------------
